# MLB Team Wins Prediction — Notebook 1: auto-sklearn

This notebook predicts MLB team wins using engineered team-season features and an `auto-sklearn` model.

## Inputs
- `data.csv`
- `data_year_team_franchise.csv`
- `predict.csv`
- `predict_year_team_franchise.csv`
- `sample_submission.csv`

## Outputs
- `prediction_submission_1.csv`

## Environment
Use the dedicated `ml2_autosklearn` environment.


In [1]:
# ============================================================
# COMMON UTILITIES
# Re-run this cell before running any model section.
# ============================================================

from __future__ import annotations

import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "data.csv"
PREDICT_PATH = DATA_DIR / "predict.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

def validate_input_files() -> None:
    required = [TRAIN_PATH, PREDICT_PATH, SUBMISSION_PATH]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required files: " + ", ".join(missing)
        )

def load_base_data():
    validate_input_files()
    train_df = pd.read_csv(TRAIN_PATH)
    pred_df = pd.read_csv(PREDICT_PATH)
    submission_df = pd.read_csv(SUBMISSION_PATH)
    return train_df, pred_df, submission_df

def _safe_divide(a, b):
    a = pd.Series(a)
    b = pd.Series(b)
    out = np.where(pd.to_numeric(b, errors="coerce") == 0, np.nan, a / b)
    return pd.Series(out, index=a.index)

def add_engineered_features(train_df: pd.DataFrame, pred_df: pd.DataFrame):
    '''
    Create era-aware team features using only variables available in the uploaded files.
    The function returns transformed copies of train and prediction data.
    '''
    train = train_df.copy()
    pred = pred_df.copy()

    combined = pd.concat(
        [train.assign(_dataset="train"), pred.assign(_dataset="predict")],
        axis=0,
        ignore_index=True,
        sort=False
    )

    # Standard inning conversion
    combined["IP"] = combined["IPouts"] / 3.0

    # --- Offensive metrics derivable from supplied data ---
    combined["TB"] = (
        combined["H"]
        + combined["2B"]
        + 2 * combined["3B"]
        + 3 * combined["HR"]
    )
    combined["OBP_simplified"] = _safe_divide(combined["H"] + combined["BB"], combined["AB"] + combined["BB"])
    combined["SLG"] = _safe_divide(combined["TB"], combined["AB"])
    combined["OPS"] = combined["OBP_simplified"] + combined["SLG"]
    combined["RunsCreated_basic"] = _safe_divide(
        (combined["H"] + combined["BB"]) * combined["TB"],
        (combined["AB"] + combined["BB"])
    )
    combined["PA_approx"] = combined["AB"] + combined["BB"]
    combined["K_rate_approx"] = _safe_divide(combined["SO"], combined["PA_approx"])
    combined["BB_rate_approx"] = _safe_divide(combined["BB"], combined["PA_approx"])
    combined["HR_rate_approx"] = _safe_divide(combined["HR"], combined["PA_approx"])
    combined["SB_rate_approx"] = _safe_divide(combined["SB"], combined["PA_approx"])

    # --- Pitching metrics derivable from supplied data ---
    combined["WHIP_calc"] = _safe_divide(combined["BBA"] + combined["HA"], combined["IP"])
    combined["H_per_9"] = _safe_divide(combined["HA"] * 9.0, combined["IP"])
    combined["BB_per_9"] = _safe_divide(combined["BBA"] * 9.0, combined["IP"])
    combined["SOA_per_9"] = _safe_divide(combined["SOA"] * 9.0, combined["IP"])
    combined["HRA_per_9"] = _safe_divide(combined["HRA"] * 9.0, combined["IP"])

    # --- Defensive / team context ---
    combined["FieldingPct"] = combined["FP"]
    combined["DP_per_game"] = _safe_divide(combined["DP"], combined["G"])
    combined["Errors_per_game"] = _safe_divide(combined["E"], combined["G"])

    # --- Team quality / context features ---
    combined["R_per_game"] = _safe_divide(combined["R"], combined["G"])
    combined["RA_per_game"] = _safe_divide(combined["RA"], combined["G"])
    combined["RunDiff"] = combined["R"] - combined["RA"]
    combined["RunDiff_per_game"] = _safe_divide(combined["RunDiff"], combined["G"])
    combined["PythagWinPct"] = _safe_divide(
        combined["R"] ** 2,
        (combined["R"] ** 2) + (combined["RA"] ** 2)
    )
    combined["WinPct_proxy"] = _safe_divide(combined.get("W", np.nan), combined["G"])

    # --- Era-aware normalization by season ---
    season_base_cols = [
        "R_per_game", "RA_per_game", "RunDiff_per_game", "OBP_simplified", "SLG",
        "OPS", "RunsCreated_basic", "K_rate_approx", "BB_rate_approx",
        "HR_rate_approx", "SB_rate_approx", "WHIP_calc", "H_per_9", "BB_per_9",
        "SOA_per_9", "HRA_per_9", "FieldingPct", "DP_per_game", "Errors_per_game",
        "PythagWinPct", "ERA", "SV"
    ]

    season_means = combined.groupby("yearID")[season_base_cols].transform("mean")
    season_stds = combined.groupby("yearID")[season_base_cols].transform("std").replace(0, np.nan)

    for col in season_base_cols:
        combined[f"{col}_year_ratio"] = _safe_divide(combined[col], season_means[col])
        combined[f"{col}_year_z"] = _safe_divide(combined[col] - season_means[col], season_stds[col])

    # Preserve only model-usable fields plus identifiers
    train_out = combined.loc[combined["_dataset"] == "train"].drop(columns=["_dataset"])
    pred_out = combined.loc[combined["_dataset"] == "predict"].drop(columns=["_dataset"])

    return train_out, pred_out

def build_feature_lists(train_df: pd.DataFrame):
    '''
    Return numeric and categorical feature lists after excluding leakage / identifiers.
    '''
    target = "W"
    exclude_cols = {
        target,
        "ID",
        "teamID",
        "year_label",
        "decade_label",
        "win_bins",
        "WinPct_proxy",   # target-derived, so exclude
    }

    categorical_cols = []
    if "franchID" in train_df.columns:
        categorical_cols.append("franchID")

    # Keep yearID because baseball run environment changes across eras.
    numeric_cols = [
        c for c in train_df.columns
        if c not in exclude_cols | set(categorical_cols)
        and pd.api.types.is_numeric_dtype(train_df[c])
    ]

    return numeric_cols, categorical_cols

def build_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor

def get_xy(train_df: pd.DataFrame, pred_df: pd.DataFrame):
    numeric_cols, categorical_cols = build_feature_lists(train_df)
    feature_cols = numeric_cols + categorical_cols

    X_train = train_df[feature_cols].copy()
    y_train = train_df["W"].copy()
    X_pred = pred_df[feature_cols].copy()

    def clean_df(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Convert boolean dtypes to integer flags for auto-sklearn compatibility
        bool_cols = df.select_dtypes(include=["bool"]).columns
        if len(bool_cols) > 0:
            df[bool_cols] = df[bool_cols].astype("int8")

        nullable_bool_cols = [c for c in df.columns if str(df[c].dtype) == "boolean"]
        if nullable_bool_cols:
            df[nullable_bool_cols] = df[nullable_bool_cols].astype("int8")

        # Convert categorical columns to numeric codes
        cat_cols = df.select_dtypes(include=["category"]).columns
        for c in cat_cols:
            df[c] = df[c].cat.codes.astype("int32")

        # Coerce object columns to numeric where possible
        obj_cols = df.select_dtypes(include=["object"]).columns
        for c in obj_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        return df

    X_train = clean_df(X_train)
    X_pred = clean_df(X_pred)

    # Enforce identical prediction columns and order
    X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)

    return X_train, y_train, X_pred, numeric_cols, categorical_cols

def evaluate_model_cv(estimator, X, y, groups=None, n_splits=5):
    '''
    Returns positive MAE values (lower is better).
    '''
    if groups is not None:
        cv = GroupKFold(n_splits=n_splits)
        scores = cross_val_score(
            estimator, X, y,
            cv=cv.split(X, y, groups=groups),
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )
    else:
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
        scores = cross_val_score(
            estimator, X, y,
            cv=cv,
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )
    return -scores

def write_submission(pred_values, submission_template, out_path):
    sub = submission_template.copy()
    sub["W"] = np.clip(np.rint(pred_values), 0, 162).astype(int)
    sub.to_csv(out_path, index=False)
    return sub

def prepare_datasets():
    train_raw, pred_raw, submission = load_base_data()
    train_fe, pred_fe = add_engineered_features(train_raw, pred_raw)
    X_train, y_train, X_pred, numeric_cols, categorical_cols = get_xy(train_fe, pred_fe)
    groups = train_fe["yearID"]
    return {
        "train_raw": train_raw,
        "pred_raw": pred_raw,
        "train_fe": train_fe,
        "pred_fe": pred_fe,
        "X_train": X_train,
        "y_train": y_train,
        "X_pred": X_pred,
        "submission": submission,
        "groups": groups,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
    }

print("Utilities loaded. Ready to run any section.")

/home/s_sofian/miniconda3/envs/mlsk/lib/python3.9/site-packages/sklearn/utils/fixes.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version  # type: ignore


Utilities loaded. Ready to run any section.


## Section 1 — Model 1: `auto-sklearn`

This section uses `AutoSklearnRegressor` when the package is available.

### Adjustable hyperparameters
Edit the config block in the next cell:
- `AUTO_SKLEARN_TIME_LEFT_FOR_THIS_TASK` — total search budget in seconds.
- `AUTO_SKLEARN_PER_RUN_TIME_LIMIT` — cap per model training attempt.
- `AUTO_SKLEARN_MEMORY_LIMIT_MB` — memory budget.
- `AUTO_SKLEARN_ENSEMBLE_SIZE` — ensemble size.
- `AUTO_SKLEARN_RESAMPLING_FOLDS` — CV folds used by auto-sklearn.
- `AUTO_SKLEARN_N_JOBS` — parallel workers.

### Why this matters
A **hyperparameter** is a setting chosen before training starts.  
Real-life metaphor: it is like deciding the oven temperature and bake time before making bread. Those settings strongly affect the final result.

In [2]:
# ============================================================
# MODEL 1 CONFIGURATION — ADJUST THESE HYPERPARAMETERS
# ============================================================

AUTO_SKLEARN_TIME_LEFT_FOR_THIS_TASK = 5400      # total search time budget in seconds
AUTO_SKLEARN_PER_RUN_TIME_LIMIT = 180           # max seconds per candidate model
AUTO_SKLEARN_MEMORY_LIMIT_MB = 8192             # memory budget for auto-sklearn
AUTO_SKLEARN_ENSEMBLE_SIZE = 40                 # larger can improve quality but costs time
AUTO_SKLEARN_RESAMPLING_FOLDS = 5               # cross-validation folds inside auto-sklearn
AUTO_SKLEARN_N_JOBS = 4                         # set >1 only if your environment supports it
AUTO_SKLEARN_TMP_FOLDER = "autosklearn_tmp"      # base temp folder name; notebook will create a fresh run-specific folder
AUTO_SKLEARN_OUTPUT_FOLDER = "autosklearn_out"   # base output folder name; notebook will create a fresh run-specific folder
AUTO_SKLEARN_CLEAN_PREVIOUS_RUN_FOLDERS = True   # set False to keep prior folders
AUTO_SKLEARN_OUTPUT_FILE = "prediction_submission_1.csv"

In [ ]:
# ============================================================
# MODEL 1 EXECUTION
# ============================================================

data_bundle = prepare_datasets()
X_train = data_bundle["X_train"]
y_train = data_bundle["y_train"]
X_pred = data_bundle["X_pred"]
submission = data_bundle["submission"]

try:
    import autosklearn.regression
    from autosklearn.metrics import mean_absolute_error as autosklearn_mae
    AUTO_SKLEARN_AVAILABLE = True
except Exception as exc:
    AUTO_SKLEARN_AVAILABLE = False
    auto_sklearn_import_error = exc

if not AUTO_SKLEARN_AVAILABLE:
    print("auto-sklearn is not installed in the active environment.")
    print("Section 1 skipped cleanly.")
    print("Import error:", repr(auto_sklearn_import_error))
    print("To run this section, install a compatible auto-sklearn build in a separate environment.")
else:
    import os
    import shutil
    import time

    run_id = time.strftime("%Y%m%d_%H%M%S")
    tmp_folder = f"{AUTO_SKLEARN_TMP_FOLDER}_{run_id}"
    output_folder = f"{AUTO_SKLEARN_OUTPUT_FOLDER}_{run_id}"

    if AUTO_SKLEARN_CLEAN_PREVIOUS_RUN_FOLDERS:
        for base_folder in [AUTO_SKLEARN_TMP_FOLDER, AUTO_SKLEARN_OUTPUT_FOLDER]:
            if os.path.exists(base_folder):
                shutil.rmtree(base_folder, ignore_errors=True)

    # Also remove the run-specific folders if they somehow already exist
    for folder in [tmp_folder, output_folder]:
        if os.path.exists(folder):
            shutil.rmtree(folder, ignore_errors=True)

    print("Using auto-sklearn tmp folder:", tmp_folder)
    print("Using auto-sklearn output folder:", output_folder)

    automl = autosklearn.regression.AutoSklearnRegressor(
        time_left_for_this_task=AUTO_SKLEARN_TIME_LEFT_FOR_THIS_TASK,
        per_run_time_limit=AUTO_SKLEARN_PER_RUN_TIME_LIMIT,
        metric=autosklearn_mae,
        ensemble_size=AUTO_SKLEARN_ENSEMBLE_SIZE,
        resampling_strategy="cv",
        resampling_strategy_arguments={"folds": AUTO_SKLEARN_RESAMPLING_FOLDS},
        memory_limit=AUTO_SKLEARN_MEMORY_LIMIT_MB,
        n_jobs=AUTO_SKLEARN_N_JOBS,
        tmp_folder=tmp_folder,
        delete_tmp_folder_after_terminate=False,
    )

    automl.fit(X_train, y_train, dataset_name="mlb_team_wins")
    train_pred = automl.predict(X_train)
    train_mae = mean_absolute_error(y_train, train_pred)

    print("Auto-sklearn training MAE:", round(train_mae, 4))
    print(automl.sprint_statistics())

    pred_values = automl.predict(X_pred)
    pred_values = np.clip(pred_values, 0, 162)

    submission = submission.copy()
    target_col = submission.columns[-1]
    submission[target_col] = pred_values
    submission.to_csv(AUTO_SKLEARN_OUTPUT_FILE, index=False)

    print(f"Saved predictions to {AUTO_SKLEARN_OUTPUT_FILE}")


Using auto-sklearn tmp folder: autosklearn_tmp_20260403_105147
Using auto-sklearn output folder: autosklearn_out_20260403_105147


/home/s_sofian/miniconda3/envs/mlsk/lib/python3.9/site-packages/sklearn/utils/fixes.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version  # type: ignore


[WARNING] [2026-04-03 10:54:54,329:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2026-04-03 10:54:54,650:Client-EnsembleBuilder] No runs were available to build an ensemble from


In [4]:
automl.show_models()

{22: {'model_id': 22,
  'rank': 1,
  'cost': 3.0258994597209736,
  'ensemble_weight': 0.04,
  'voting_model': VotingRegressor(estimators=None),
  'estimators': [{'data_preprocessor': <autosklearn.pipeline.components.data_preprocessing.DataPreprocessorChoice at 0x79fdbd7c4eb0>,
    'feature_preprocessor': <autosklearn.pipeline.components.feature_preprocessing.FeaturePreprocessorChoice at 0x79fe0b07f580>,
    'regressor': <autosklearn.pipeline.components.regression.RegressorChoice at 0x79fdbdbc50a0>,
    'sklearn_regressor': HistGradientBoostingRegressor(l2_regularization=1.8428972335335263e-10,
                                  learning_rate=0.012607824914758717, max_iter=512,
                                  max_leaf_nodes=10, min_samples_leaf=8,
                                  n_iter_no_change=0, random_state=1,
                                  validation_fraction=None, warm_start=True)},
   {'data_preprocessor': <autosklearn.pipeline.components.data_preprocessing.DataPreprocessor

## Output check

In [5]:
expected_files = ['prediction_submission_1.csv']
for f in expected_files:
    print(f, 'exists' if Path(f).exists() else 'missing')

prediction_submission_1.csv exists
